# Amazon ML Challenge 2026: Business Entity Resolution
## GPU-Accelerated End-to-End Pipeline (Kaggle Edition)

This notebook runs the complete 7-channel blocking, 40-feature engineering, GPU-accelerated Ensemble model (LightGBM + XGBoost), threshold calibration for Macro $F_{0.5}$, and official submission validation.

In [ ]:
# 1. Clone repository and install dependencies
import os
import sys

!git clone https://github.com/Aviral2406/MLchLg.git
%cd MLchLg
!git pull origin main

!pip install -q rapidfuzz lightgbm xgboost pyyaml scikit-learn

In [ ]:
# 2. Locate Dataset Paths (Supports Kaggle Input or local repo data)
from pathlib import Path

kaggle_input = Path('/kaggle/input')
dataset_dir = None

if kaggle_input.exists():
    for p in kaggle_input.glob('**/*'):
        if (p / 'train_source1.tsv').exists():
            dataset_dir = p.parent
            break
        if (p / 'train' / 'train_source1.tsv').exists():
            dataset_dir = p
            break

if dataset_dir is None:
    dataset_dir = Path('data')

train_dir = dataset_dir / 'train' if (dataset_dir / 'train').exists() else dataset_dir
test_dir = dataset_dir / 'test' if (dataset_dir / 'test').exists() else dataset_dir

print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")

In [ ]:
# 3. Verify GPU Acceleration
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. Run Training & Threshold Tuning on Kaggle GPU
# This uses the 7-channel blocking and 40 pairwise features with XGBoost/LightGBM GPU mode
!python scripts/train_and_evaluate.py

In [ ]:
# 5. Run Official Submission Validator
!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir {test_dir}

In [ ]:
# 6. Package Submission for Download
import shutil
shutil.make_archive('/kaggle/working/submission', 'zip', 'output')
print("Submission package created at /kaggle/working/submission.zip!")
print("You can download it directly from the Kaggle Output panel on the right sidebar.")

### 7. (Optional) Push Results back to GitHub so Antigravity can inspect them
Uncomment the cell below and add your GitHub Personal Access Token to push trained experiment logs and model weights directly back to the repo!

In [ ]:
# GITHUB_TOKEN = 'YOUR_GITHUB_PERSONAL_ACCESS_TOKEN'
# !git config --global user.name 'Aviral2406'
# !git config --global user.email 'aviral@example.com'
# !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Aviral2406/MLchLg.git
# !git add output/ experiments/ configs/
# !git commit -m 'Add trained model outputs from Kaggle GPU'
# !git push origin main